# Lesson 9 — Capstone: Running on Real Hardware

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 9. Work through the cells in order. Every cell is meant to be run; real-hardware cells are commented out and require an active IBM Quantum account.

**Topics covered:**
- Transpilation: stages, optimization levels, circuit depth analysis
- Native gate sets and basis translation
- Device topology and SWAP routing
- Submitting jobs via SamplerV2 (fake and real backends)
- Readout error calibration and correction
- Capstone project: run an algorithm on real hardware and analyse the results

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

In [ ]:
# Core imports and Grover helper functions from Lesson 8
# (reproduced here so this notebook is self-contained)

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt


# --- Grover functions (Lesson 8) ---

def phase_oracle(n, target):
    """
    Phase oracle: flips the sign of |target> and leaves all other states unchanged.
    target: binary string in Qiskit bitstring order (target[0] = MSB = highest qubit).
    """
    oracle = QuantumCircuit(n, name='Oracle')
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)
    oracle.h(n - 1)
    oracle.mcx(list(range(n - 1)), n - 1)
    oracle.h(n - 1)
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)
    return oracle


def diffuser(n):
    """Grover diffuser: D = 2|s><s| - I where |s> = H^n|0>."""
    diff = QuantumCircuit(n, name='Diffuser')
    diff.h(range(n))
    diff.x(range(n))
    diff.h(n - 1)
    diff.mcx(list(range(n - 1)), n - 1)
    diff.h(n - 1)
    diff.x(range(n))
    diff.h(range(n))
    return diff


def grover_circuit(n, target, num_iterations):
    """Build Grover's circuit (no measurement)."""
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.barrier()
    oracle = phase_oracle(n, target)
    diff = diffuser(n)
    for _ in range(num_iterations):
        qc.compose(oracle, inplace=True)
        qc.compose(diff, inplace=True)
        qc.barrier()
    return qc


def optimal_iterations(n):
    """Optimal Grover iteration count for N=2^n, one marked item."""
    N = 2**n
    theta = np.arcsin(1 / np.sqrt(N))
    return round(np.pi / (4 * theta) - 0.5)


print("Grover helper functions defined.")

## 1. The Transpilation Pipeline

Transpilation converts an abstract Qiskit circuit into one that a specific device can execute. It runs in five stages: **Init** (preprocessing), **Layout** (assigning logical to physical qubits), **Routing** (inserting SWAP gates for non-adjacent pairs), **Translation** (converting to the native gate set), and **Optimization** (algebraic simplification).

The `optimization_level` parameter (0 to 3) controls aggressiveness. Level 1 is the default; level 3 runs slower but reduces two-qubit gate count, which matters most for fidelity because two-qubit gates are roughly ten times noisier than single-qubit gates.

In [ ]:
# Set up the fake backend
backend = FakeSherbrooke()                         # 127-qubit fake device with calibrated noise
noisy_sim = AerSimulator.from_backend(backend)

print(f"Backend: {backend.name}")
print(f"Qubits:  {backend.num_qubits}")

In [ ]:
# Build the Grover circuit (n=3, k=2, target='101') and compare optimization levels
n, target, k = 3, '101', 2
qc_raw = grover_circuit(n, target, num_iterations=k)
qc_raw.measure_all()

print(f"Raw circuit: depth={qc_raw.depth()}, ops={dict(qc_raw.count_ops())}")
print()

print(f"{'Level':>6}  {'Depth':>6}  {'2Q gates':>9}  {'Total ops':>10}")
print("-" * 38)
for level in range(4):
    t = transpile(qc_raw, backend=backend, optimization_level=level, seed_transpiler=42)
    two_q = sum(t.count_ops().get(g, 0) for g in ['cx', 'ecr', 'cz'])
    total = sum(t.count_ops().values())
    print(f"     {level}  {t.depth():>6}  {two_q:>9}  {total:>10}")

In [ ]:
# Draw the transpiled circuit (level 3) to see the native gate decomposition
t3 = transpile(qc_raw, backend=backend, optimization_level=3, seed_transpiler=42)
print(f"Level 3 transpiled circuit:")
print(f"  Depth: {t3.depth()}")
print(f"  Ops:   {dict(t3.count_ops())}")
print(f"  Layout (logical -> physical): ", end='')
if t3.layout is not None and t3.layout.initial_layout is not None:
    layout = t3.layout.initial_layout
    for lq in qc_raw.qubits:
        phys = layout[lq]
        print(f"q{qc_raw.qubits.index(lq)}->{phys}", end='  ')
print()

## 2. Basis Gate Sets

IBM hardware exposes a small set of native gates:
- **`rz(θ)`**: virtual rotation about Z. Zero pulse duration, zero error contribution.
- **`sx`**: square root of X. One physical pulse.
- **`x`**: Pauli X. One physical pulse.
- **`ecr`** or **`cx`**: the native two-qubit entangling gate.

Every other gate in your circuit is decomposed into these. The Toffoli gate (CCX), for example, requires six CX gates after decomposition — which is why circuits with multi-controlled gates grow expensive quickly.

In [ ]:
# Inspect the native gate set
print("Native operations:", backend.operation_names)
print()

# T gate: should become a single rz (virtual, zero error)
qc_t = QuantumCircuit(1)
qc_t.t(0)
t_t = transpile(qc_t, backend=backend, optimization_level=1, seed_transpiler=0)
print(f"T gate transpiled ops:   {dict(t_t.count_ops())}")

# H gate: should become rz + sx + rz (one physical pulse)
qc_h = QuantumCircuit(1)
qc_h.h(0)
t_h = transpile(qc_h, backend=backend, optimization_level=1, seed_transpiler=0)
print(f"H gate transpiled ops:   {dict(t_h.count_ops())}")

# S gate: should be a single rz(pi/2) — virtual
qc_s = QuantumCircuit(1)
qc_s.s(0)
t_s = transpile(qc_s, backend=backend, optimization_level=1, seed_transpiler=0)
print(f"S gate transpiled ops:   {dict(t_s.count_ops())}")

print()

# CCX (Toffoli): the most expensive common gate
qc_toff = QuantumCircuit(3)
qc_toff.ccx(0, 1, 2)
t_toff = transpile(qc_toff, backend=backend, optimization_level=3, seed_transpiler=0)
print(f"CCX transpiled: depth={t_toff.depth()}, ops={dict(t_toff.count_ops())}")

In [ ]:
# Compare the physical pulse cost of common gates
# Only sx, x, and the two-qubit gate contribute to decoherence.
gates_to_check = [('T', lambda qc: qc.t(0), 1),
                  ('S', lambda qc: qc.s(0), 1),
                  ('H', lambda qc: qc.h(0), 1),
                  ('RX(π/3)', lambda qc: qc.rx(np.pi/3, 0), 1),
                  ('CX', lambda qc: qc.cx(0, 1), 2),]

print(f"{'Gate':>10}  {'rz (virtual)':>13}  {'sx (pulse)':>11}  {'cx/ecr (pulse)':>15}")
print("-" * 56)
for name, apply_fn, n_q in gates_to_check:
    qc_g = QuantumCircuit(n_q)
    apply_fn(qc_g)
    t_g = transpile(qc_g, backend=backend, optimization_level=1, seed_transpiler=0)
    ops = t_g.count_ops()
    rz = ops.get('rz', 0)
    sx = ops.get('sx', 0)
    two_q = ops.get('cx', 0) + ops.get('ecr', 0)
    print(f"{name:>10}  {rz:>13}  {sx:>11}  {two_q:>15}")

## 3. Device Topology and Routing

IBM processors use a **heavy-hex lattice**: a hexagonal-style grid where each qubit connects to two or three neighbors. When a circuit requests a two-qubit gate between non-adjacent qubits, the router inserts SWAP gates. Each SWAP decomposes into three CX gates, so a poor layout multiplies circuit depth and noise.

SABRE layout minimises expected SWAP count by finding a placement where frequently interacting qubits are already adjacent.

In [ ]:
# Examine the coupling map
# edge_list() returns (source, target) tuples; edges() returns only edge weights
cm = backend.coupling_map
edges = list(cm.graph.edge_list())

print(f"Backend: {backend.name}")
print(f"Total qubits: {backend.num_qubits}")
print(f"Coupling edges: {len(edges)}")
print(f"Average qubit degree: {2 * len(edges) / backend.num_qubits:.2f}")
print(f"\nFirst 15 connected pairs:")
for u, v in edges[:15]:
    print(f"  {u:>3} -- {v:>3}")

In [ ]:
# Demonstrate SWAP overhead from a bad layout
qc_test = QuantumCircuit(2)
qc_test.h(0)
qc_test.cx(0, 1)

# On heavy-hex, qubits 0 and 2 are typically not adjacent (0-1 and 1-2 are)
t_bad = transpile(qc_test, backend=backend, initial_layout=[0, 2],
                  optimization_level=1, seed_transpiler=42)
print("CX(0,1) with forced non-adjacent layout [0, 2]:")
print(f"  Depth: {t_bad.depth()}, Ops: {dict(t_bad.count_ops())}")
print()

# SABRE finds an adjacent placement automatically
t_good = transpile(qc_test, backend=backend, optimization_level=1, seed_transpiler=42)
print("CX(0,1) with SABRE layout:")
print(f"  Depth: {t_good.depth()}, Ops: {dict(t_good.count_ops())}")
print()
print("SABRE eliminates the SWAP entirely by finding an adjacent placement.")

In [ ]:
# Show how SWAP overhead grows with circuit width when layout is forced to be bad
# For a linear chain of CX gates: CX(0,1), CX(1,2), CX(2,3), ...
# forcing a reversed layout (n-1, n-2, ..., 0) maximises routing overhead

print(f"{'n':>4}  {'good layout depth':>18}  {'reversed layout depth':>21}  {'overhead':>9}")
print("-" * 58)

for n_chain in range(2, 7):
    qc_chain = QuantumCircuit(n_chain)
    for i in range(n_chain - 1):
        qc_chain.cx(i, i + 1)

    t_fwd = transpile(qc_chain, backend=backend, optimization_level=1, seed_transpiler=0)
    rev_layout = list(range(n_chain - 1, -1, -1))
    t_rev = transpile(qc_chain, backend=backend, initial_layout=rev_layout,
                      optimization_level=1, seed_transpiler=0)

    print(f"{n_chain:>4}  {t_fwd.depth():>18}  {t_rev.depth():>21}  "
          f"{t_rev.depth() / t_fwd.depth():.2f}x")

## 4. Submitting Jobs

You saved IBM Quantum credentials in Lesson 1. Real-hardware submission uses `QiskitRuntimeService` and the `SamplerV2` primitive. The cells below run on the fake backend; uncomment the real-hardware blocks when you have an active account and are ready to queue.

In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

try:
    token = userdata.get("IBM_TOKEN")
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        overwrite=True
    )
    print("Account saved successfully.")
except Exception as e:
    print("Could not load IBM_TOKEN from secrets:", e)
    print("Use Option B below instead.")

In [ ]:
# --- Connecting to IBM Quantum (real hardware) ---
# Uncomment when you have saved credentials and want to use real hardware.

# from qiskit_ibm_runtime import QiskitRuntimeService
#
# service = QiskitRuntimeService()      # loads saved token
# backend_real = service.least_busy(operational=True, simulator=False)
# print(f"Backend: {backend_real.name}")
# print(f"Qubits: {backend_real.num_qubits}")
# print(f"Pending jobs: {backend_real.status().pending_jobs}")

print("Using FakeSherbrooke for local noisy simulation (no account required).")

In [ ]:
# Run the Grover circuit on the fake noisy backend
initial_layout = [0, 1, 2]   # fix physical qubits for reproducibility

t_grover = transpile(qc_raw, backend=backend, initial_layout=initial_layout,
                     optimization_level=3, seed_transpiler=42)

print(f"Transpiled circuit:")
print(f"  Depth:          {t_grover.depth()}")
print(f"  Two-qubit gates: {sum(t_grover.count_ops().get(g, 0) for g in ['cx', 'ecr'])}")
print()

job = noisy_sim.run(t_grover, shots=4096)
counts_noisy = job.result().get_counts()

top = max(counts_noisy, key=counts_noisy.get)
print(f"Noisy result:  top='{top}', P={counts_noisy.get(top, 0) / 4096:.3f}")
print(f"Target: '{target}'")
print(f"Match: {top == target}")

In [ ]:
# --- Real-hardware submission via SamplerV2 (requires IBM Quantum account) ---
# from qiskit_ibm_runtime import SamplerV2 as Sampler
#
# t_real = transpile(qc_raw, backend=backend_real, optimization_level=3)
#
# # Pass the backend directly — works on all plans including the free Open Plan.
# # Sessions require a paid plan and are not needed for single job submission.
# sampler = Sampler(backend=backend_real)
# job = sampler.run([(t_real,)], shots=4096)
# print(f"Job submitted. ID: {job.job_id()}")
# result = job.result()
# counts_real = result[0].data.meas.get_counts()
# top_real = max(counts_real, key=counts_real.get)
# print(f"Hardware result: top='{top_real}', P={counts_real[top_real] / 4096:.3f}")

print("Real-hardware block is commented out. Uncomment when ready to submit.")

## 5. Readout Error Mitigation

Readout errors happen when a qubit relaxes during measurement and registers the wrong outcome. For a single qubit, the **assignment matrix** $A$ captures these probabilities:

$$A = \begin{pmatrix} P(0 \mid 0) & P(0 \mid 1) \\ P(1 \mid 0) & P(1 \mid 1) \end{pmatrix}$$

The measured probabilities relate to the ideal by $\mathbf{p}_{\text{noisy}} = A\,\mathbf{p}_{\text{ideal}}$, so the correction is $\hat{\mathbf{p}} = A^{-1}\mathbf{p}_{\text{noisy}}$.

For $n$ qubits, assuming independent qubit errors, the full correction matrix is the tensor product of the per-qubit matrices: $A_{\text{full}} = A_{n-1} \otimes \cdots \otimes A_0$.

In [ ]:
def calibrate_qubit(sim, backend, physical_qubit, shots=8192):
    """
    Estimate the 2x2 assignment matrix for one physical qubit.
    Returns A where A[r, p] = P(read r | prepared p).
    """
    A = np.zeros((2, 2))
    for prep in range(2):
        qc_cal = QuantumCircuit(1, 1)
        if prep:
            qc_cal.x(0)
        qc_cal.measure(0, 0)
        t_cal = transpile(qc_cal, backend=backend,
                          initial_layout=[physical_qubit],
                          optimization_level=0, seed_transpiler=0)
        counts = sim.run(t_cal, shots=shots).result().get_counts()
        A[0, prep] = counts.get('0', 0) / shots   # P(read 0 | prepared prep)
        A[1, prep] = counts.get('1', 0) / shots   # P(read 1 | prepared prep)
    return A


# Calibrate the three physical qubits used by the transpiled circuit
cal_matrices = [calibrate_qubit(noisy_sim, backend, q) for q in initial_layout]

print("Assignment matrices (A[r, p] = P(read r | prepared p)):\n")
for q, A in zip(initial_layout, cal_matrices):
    print(f"  Physical qubit {q}:")
    print(f"    P(0|0)={A[0,0]:.4f}  P(0|1)={A[0,1]:.4f}")
    print(f"    P(1|0)={A[1,0]:.4f}  P(1|1)={A[1,1]:.4f}")
    print()

In [ ]:
def apply_readout_correction(counts, cal_matrices, n_qubits, shots):
    """
    Apply local (independent per-qubit) readout correction.
    cal_matrices[i] is the 2x2 matrix for logical qubit i
    (qubit 0 = LSB = rightmost character in the bitstring).

    Full correction matrix: A[n-1] ⊗ A[n-2] ⊗ ... ⊗ A[0]
    (qubit n-1 = MSB = leftmost character in the bitstring).
    """
    # Build full 2^n x 2^n tensor product, MSB first
    A_full = cal_matrices[n_qubits - 1]
    for i in range(n_qubits - 2, -1, -1):
        A_full = np.kron(A_full, cal_matrices[i])

    # Build probability vector from counts
    # int(bitstr, 2) gives the correct index (MSB left, LSB right)
    N = 2**n_qubits
    p_noisy = np.zeros(N)
    for bitstr, cnt in counts.items():
        p_noisy[int(bitstr, 2)] = cnt / shots

    # Correct and clip small negatives from matrix inversion
    p_corrected = np.linalg.inv(A_full) @ p_noisy
    p_corrected = np.clip(p_corrected, 0, None)
    if p_corrected.sum() > 0:
        p_corrected /= p_corrected.sum()

    return {format(i, f'0{n_qubits}b'): float(p_corrected[i]) for i in range(N)}


# Apply correction to the noisy Grover counts
corrected = apply_readout_correction(counts_noisy, cal_matrices, n, shots=4096)

p_noisy_target     = counts_noisy.get(target, 0) / 4096
p_corrected_target = corrected.get(target, 0)

print(f"P('{target}') noisy:     {p_noisy_target:.3f}")
print(f"P('{target}') corrected: {p_corrected_target:.3f}")
print()
print("Readout correction recovers some probability.")
print("The remaining gap relative to ideal (0.945) is gate noise.")

## 6. Simulator vs Hardware: Full Comparison

Three probability distributions placed side by side reveal the distinct contributions of routing overhead, gate noise, and readout noise.

In [ ]:
# Ideal (noiseless) probabilities from the statevector
ideal_probs = Statevector(grover_circuit(n, target, k)).probabilities_dict()

states         = sorted(ideal_probs.keys())
ideal_vals     = [ideal_probs.get(s, 0) for s in states]
noisy_vals     = [counts_noisy.get(s, 0) / 4096 for s in states]
corrected_vals = [corrected.get(s, 0) for s in states]

x = np.arange(len(states))
width = 0.28

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width, ideal_vals,     width, label='Ideal (no noise)',     color='steelblue')
ax.bar(x,         noisy_vals,     width, label='Noisy (transpiled)',   color='tomato')
ax.bar(x + width, corrected_vals, width, label='Readout-corrected',    color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels([f'|{s}⟩' for s in states])
ax.set_ylabel('Probability')
ax.set_title(f"Grover n={n}, target='{target}', k={k}: ideal vs noisy vs corrected")
ax.axhline(1 / 2**n, color='gray', linestyle=':', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSummary for target='{target}':")
print(f"  Ideal:     {ideal_probs.get(target, 0):.3f}")
print(f"  Noisy:     {p_noisy_target:.3f}")
print(f"  Corrected: {p_corrected_target:.3f}")

## 7. Capstone Project

Choose **one** algorithm:
- **Deutsch-Jozsa** with $n = 4$ qubits (constant or balanced oracle)
- **Bernstein-Vazirani** with a secret string of your choice ($n \geq 4$)
- **Grover's algorithm** with $n = 3$ or $n = 4$ and a target of your choice

Complete all six steps in the scaffold below.

In [ ]:
# ============================================================
# CAPSTONE PROJECT SCAFFOLD
# Fill in each TODO section.
# ============================================================

# --- Choose your algorithm and parameters ---
# Grover example (edit or replace with DJ / BV as desired)
cap_n      = 3
cap_target = '011'   # change to any n-bit string
cap_k      = optimal_iterations(cap_n)

print(f"Algorithm: Grover  n={cap_n}  target='{cap_target}'  k={cap_k}")

In [ ]:
# Step 1: Baseline simulation (ideal, no noise)
# TODO: build the circuit, run on AerSimulator(), record P(correct answer)

cap_qc = grover_circuit(cap_n, cap_target, cap_k)
cap_qc.measure_all()

ideal_sim = AerSimulator()
t_ideal_cap = transpile(cap_qc, backend=ideal_sim, optimization_level=0)
counts_ideal_cap = ideal_sim.run(t_ideal_cap, shots=4096).result().get_counts()

p_ideal_cap = counts_ideal_cap.get(cap_target, 0) / 4096
print(f"Step 1 — Ideal P('{cap_target}'): {p_ideal_cap:.3f}")

In [ ]:
# Step 2: Transpilation analysis
# TODO: transpile at levels 1 and 3, report depth and two-qubit gate count

print("Step 2 — Transpilation analysis:\n")
print(f"  Raw circuit: depth={cap_qc.depth()}")
print()

for level in [1, 3]:
    t_cap = transpile(cap_qc, backend=backend, optimization_level=level, seed_transpiler=42)
    two_q_cap = sum(t_cap.count_ops().get(g, 0) for g in ['cx', 'ecr', 'cz'])
    print(f"  Level {level}: depth={t_cap.depth()}, two-qubit gates={two_q_cap}")
    print(f"         ops: {dict(t_cap.count_ops())}")
    print()

In [ ]:
# Step 3: Noisy simulation
# TODO: run the level-3 transpiled circuit on noisy_sim, record P(correct)

cap_layout = list(range(cap_n))   # physical qubits 0..n-1 (adjacent on heavy-hex for small n)
t_cap_l3 = transpile(cap_qc, backend=backend, initial_layout=cap_layout,
                     optimization_level=3, seed_transpiler=42)

counts_noisy_cap = noisy_sim.run(t_cap_l3, shots=4096).result().get_counts()
p_noisy_cap = counts_noisy_cap.get(cap_target, 0) / 4096

print(f"Step 3 — Noisy P('{cap_target}'): {p_noisy_cap:.3f}")

In [ ]:
# Step 4: Readout mitigation
# TODO: calibrate qubits, apply correction, record corrected P(correct)

cal_matrices_cap = [calibrate_qubit(noisy_sim, backend, q) for q in cap_layout]
corrected_cap = apply_readout_correction(counts_noisy_cap, cal_matrices_cap, cap_n, shots=4096)

p_corrected_cap = corrected_cap.get(cap_target, 0)
print(f"Step 4 — Corrected P('{cap_target}'): {p_corrected_cap:.3f}")

In [ ]:
# Step 5: Real hardware (optional — requires IBM Quantum account)
# Uncomment and fill in when ready to submit.
#
# from qiskit_ibm_runtime import SamplerV2 as Sampler
#
# t_real_cap = transpile(cap_qc, backend=backend_real, optimization_level=3)
#
# # Pass the backend directly — works on all plans including the free Open Plan.
# # Sessions require a paid plan and are not needed for single job submission.
# sampler = Sampler(backend=backend_real)
# job_real = sampler.run([(t_real_cap,)], shots=4096)
# print(f"Job ID: {job_real.job_id()}")
# result_real = job_real.result()
# counts_real_cap = result_real[0].data.meas.get_counts()
# p_real_cap = counts_real_cap.get(cap_target, 0) / 4096
# print(f"Hardware P('{cap_target}'): {p_real_cap:.3f}")

print("Step 5 — Real hardware: commented out (uncomment when ready).")

In [ ]:
# Step 6: Comparison chart
# TODO: add a 'real hardware' bar if you completed Step 5

cap_ideal_probs = Statevector(grover_circuit(cap_n, cap_target, cap_k)).probabilities_dict()

cap_states         = sorted(cap_ideal_probs.keys())
cap_ideal_vals     = [cap_ideal_probs.get(s, 0) for s in cap_states]
cap_noisy_vals     = [counts_noisy_cap.get(s, 0) / 4096 for s in cap_states]
cap_corrected_vals = [corrected_cap.get(s, 0) for s in cap_states]

x = np.arange(len(cap_states))
width = 0.28

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width, cap_ideal_vals,     width, label='Ideal',           color='steelblue')
ax.bar(x,         cap_noisy_vals,     width, label='Noisy',           color='tomato')
ax.bar(x + width, cap_corrected_vals, width, label='Corrected',       color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels([f'|{s}⟩' for s in cap_states])
ax.set_ylabel('Probability')
ax.set_title(f"Capstone: n={cap_n}, target='{cap_target}', k={cap_k}")
ax.axhline(1 / 2**cap_n, color='gray', linestyle=':', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nCapstone summary:")
print(f"  Ideal:     {cap_ideal_probs.get(cap_target, 0):.3f}")
print(f"  Noisy:     {p_noisy_cap:.3f}")
print(f"  Corrected: {p_corrected_cap:.3f}")

In [ ]:
# Step 6 (continued): Written analysis template
# Replace each placeholder with your observations.

analysis = """
Algorithm: Grover, n={cap_n}, target='{cap_target}', k={cap_k}

Transpilation:
  Original depth: [TODO]
  Level 1 depth: [TODO], two-qubit gates: [TODO]
  Level 3 depth: [TODO], two-qubit gates: [TODO]
  [Describe the overhead and what caused it.]

Noise impact:
  Ideal P(correct):     [TODO]
  Noisy P(correct):     [TODO]
  Absolute degradation: [TODO]
  [Explain why the probability dropped.]

Readout correction:
  Corrected P(correct): [TODO]
  Recovery:             [TODO]
  [Explain what the correction did and did not fix.]

Real hardware (if completed):
  Hardware P(correct): [TODO]
  Comparison with FakeSherbrooke: [TODO]
"""
print(analysis)

## Summary

In this lesson you:

- Transpiled the Grover circuit across all four optimization levels and observed how two-qubit gate count decreases from level 0 to 3, directly improving expected fidelity.
- Inspected IBM's native gate set and confirmed that `rz` is virtual (zero error), while `sx`, `x`, and `ecr`/`cx` consume pulse time and accumulate decoherence.
- Demonstrated SWAP routing overhead on the heavy-hex coupling map: a forced non-adjacent layout multiplied the CX count and depth; SABRE eliminated the overhead by finding adjacent physical qubit placement.
- Ran the transpiled circuit on a noisy simulator backed by FakeSherbrooke calibration data, observing a substantial drop in target probability relative to the ideal.
- Calibrated per-qubit assignment matrices and applied local readout correction, recovering a portion of the probability lost to measurement errors.
- Built a three-bar comparison chart showing ideal, noisy, and corrected results, making the distinct contributions of gate noise and readout noise visible.

**Qiskit II** begins where this lesson ends. Lesson 1 introduces density matrices for describing mixed states, the natural language for noisy quantum systems. Lesson 2 builds precise noise models from Kraus operators and real device data. Lesson 3 covers zero-noise extrapolation and probabilistic error cancellation: the techniques that address gate errors rather than only readout errors. By Lesson 9 you will implement Shor's factoring algorithm and estimate its resource requirements on fault-tolerant hardware.